In [ ]:
import os
import glob
import pickle
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import seaborn as sns
from pathlib import Path
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['font.size'] = 10


In [ ]:
# Additional methods for the ExperimentVisualizer class
# (These should be added to the class above)

def load_image_safely(self, image_path: Path) -> Optional[np.ndarray]:
    """Safely load an image file"""
    try:
        img = Image.open(image_path)
        return np.array(img)
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None

def load_pickle_safely(self, pickle_path: Path) -> Optional[any]:
    """Safely load a pickle file"""
    try:
        with open(pickle_path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        print(f"Error loading pickle {pickle_path}: {e}")
        return None

def plot_experiment_overview(self):
    """Create an overview plot of all experiments"""
    if not self.experiment_data:
        print("No experiment data found. Run find_matching_experiments() first.")
        return
    
    # Collect statistics
    stats = []
    for exp_id, data in self.experiment_data.items():
        stats.append({
            'Experiment ID': exp_id,
            'GSAM Images': len(data['gsam_files']['images']),
            'GSAM Pickles': len(data['gsam_files']['pickles']),
            'FLUX Images': len(data['flux_files']['images']),
            'FLUX Pickles': len(data['flux_files']['pickles']),
            'Total GSAM Files': len(data['gsam_files']['all_files']),
            'Total FLUX Files': len(data['flux_files']['all_files'])
        })
    
    df = pd.DataFrame(stats)
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Experiment Overview', fontsize=16, fontweight='bold')
    
    # Plot 1: Image counts
    ax1 = axes[0, 0]
    x_pos = np.arange(len(df))
    width = 0.35
    ax1.bar(x_pos - width/2, df['GSAM Images'], width, label='GSAM', alpha=0.8)
    ax1.bar(x_pos + width/2, df['FLUX Images'], width, label='FLUX', alpha=0.8)
    ax1.set_xlabel('Experiment ID')
    ax1.set_ylabel('Number of Images')
    ax1.set_title('Image Counts by Experiment')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(df['Experiment ID'], rotation=45)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Pickle counts
    ax2 = axes[0, 1]
    ax2.bar(x_pos - width/2, df['GSAM Pickles'], width, label='GSAM', alpha=0.8)
    ax2.bar(x_pos + width/2, df['FLUX Pickles'], width, label='FLUX', alpha=0.8)
    ax2.set_xlabel('Experiment ID')
    ax2.set_ylabel('Number of Pickle Files')
    ax2.set_title('Pickle File Counts by Experiment')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(df['Experiment ID'], rotation=45)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Total file counts
    ax3 = axes[1, 0]
    ax3.bar(x_pos - width/2, df['Total GSAM Files'], width, label='GSAM', alpha=0.8)
    ax3.bar(x_pos + width/2, df['Total FLUX Files'], width, label='FLUX', alpha=0.8)
    ax3.set_xlabel('Experiment ID')
    ax3.set_ylabel('Total Files')
    ax3.set_title('Total File Counts by Experiment')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(df['Experiment ID'], rotation=45)
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: File type distribution (pie chart for all experiments combined)
    ax4 = axes[1, 1]
    total_gsam_images = df['GSAM Images'].sum()
    total_gsam_pickles = df['GSAM Pickles'].sum()
    total_flux_images = df['FLUX Images'].sum()
    total_flux_pickles = df['FLUX Pickles'].sum()
    
    labels = ['GSAM Images', 'GSAM Pickles', 'FLUX Images', 'FLUX Pickles']
    sizes = [total_gsam_images, total_gsam_pickles, total_flux_images, total_flux_pickles]
    colors = ['lightblue', 'lightcoral', 'lightgreen', 'lightyellow']
    
    ax4.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax4.set_title('Overall File Type Distribution')
    
    plt.tight_layout()
    plt.show()
    
    # Display summary table
    print("\nExperiment Summary:")
    print(df.to_string(index=False))

# Add these methods to the ExperimentVisualizer class by copying them manually


In [ ]:
# Complete ExperimentVisualizer class with all methods
class ExperimentVisualizer:
    """A class to visualize and compare GSAM and FLUX experiments"""
    
    def __init__(self, gsam_dir: str, flux_dir: str):
        self.gsam_dir = Path(gsam_dir)
        self.flux_dir = Path(flux_dir)
        self.experiment_data = {}
        
        if not self.gsam_dir.exists():
            raise ValueError(f"GSAM directory does not exist: {gsam_dir}")
        if not self.flux_dir.exists():
            raise ValueError(f"FLUX directory does not exist: {flux_dir}")
    
    def find_matching_experiments(self) -> Dict[str, Dict]:
        """Find matching numbered directories between GSAM and FLUX experiments"""
        gsam_dirs = {d.name: d for d in self.gsam_dir.iterdir() if d.is_dir() and d.name.isdigit()}
        flux_dirs = {d.name: d for d in self.flux_dir.iterdir() if d.is_dir() and d.name.isdigit()}
        
        matching_ids = set(gsam_dirs.keys()) & set(flux_dirs.keys())
        
        self.experiment_data = {
            exp_id: {
                'gsam_path': gsam_dirs[exp_id],
                'flux_path': flux_dirs[exp_id],
                'gsam_files': self._get_files(gsam_dirs[exp_id]),
                'flux_files': self._get_files(flux_dirs[exp_id])
            }
            for exp_id in matching_ids
        }
        
        print(f"Found {len(matching_ids)} matching experiments")
        print(f"Experiment IDs: {sorted(matching_ids)}")
        return self.experiment_data
    
    def _get_files(self, directory: Path) -> Dict[str, List[Path]]:
        """Get all images and pickle files from a directory"""
        files = {'images': [], 'pickles': [], 'all_files': []}
        img_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif'}
        
        for file_path in directory.rglob('*'):
            if file_path.is_file():
                files['all_files'].append(file_path)
                if file_path.suffix.lower() in img_extensions:
                    files['images'].append(file_path)
                elif file_path.suffix.lower() == '.pkl':
                    files['pickles'].append(file_path)
        return files
    
    def load_image_safely(self, image_path: Path) -> Optional[np.ndarray]:
        """Safely load an image file"""
        try:
            img = Image.open(image_path)
            return np.array(img)
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            return None
    
    def load_pickle_safely(self, pickle_path: Path) -> Optional[any]:
        """Safely load a pickle file"""
        try:
            with open(pickle_path, 'rb') as f:
                return pickle.load(f)
        except Exception as e:
            print(f"Error loading pickle {pickle_path}: {e}")
            return None


In [ ]:
# Additional methods for detailed visualization and comparison
def plot_individual_experiment(visualizer, experiment_id: str, max_images: int = 8):
    """
    Plot images from a specific experiment for both GSAM and FLUX
    
    Args:
        visualizer: ExperimentVisualizer instance
        experiment_id: The experiment ID to visualize
        max_images: Maximum number of images to display per method
    """
    if experiment_id not in visualizer.experiment_data:
        print(f"Experiment {experiment_id} not found!")
        return
    
    data = visualizer.experiment_data[experiment_id]
    
    # Get image files
    gsam_images = data['gsam_files']['images'][:max_images]
    flux_images = data['flux_files']['images'][:max_images]
    
    if not gsam_images and not flux_images:
        print(f"No images found for experiment {experiment_id}")
        return
    
    # Calculate grid size
    max_cols = 4
    gsam_rows = (len(gsam_images) + max_cols - 1) // max_cols if gsam_images else 0
    flux_rows = (len(flux_images) + max_cols - 1) // max_cols if flux_images else 0
    total_rows = gsam_rows + flux_rows + 2  # +2 for titles
    
    fig = plt.figure(figsize=(16, 4 * total_rows))
    
    current_row = 1
    
    # Plot GSAM images
    if gsam_images:
        # GSAM title
        plt.subplot(total_rows, 1, current_row)
        plt.text(0.5, 0.5, f'GSAM Experiment {experiment_id}', 
                ha='center', va='center', fontsize=16, fontweight='bold')
        plt.axis('off')
        current_row += 1
        
        # GSAM images
        for i, img_path in enumerate(gsam_images):
            img = visualizer.load_image_safely(img_path)
            if img is not None:
                row = current_row + i // max_cols
                col = i % max_cols + 1
                plt.subplot(gsam_rows, max_cols, (i // max_cols) * max_cols + col)
                plt.imshow(img)
                plt.title(f'GSAM: {img_path.name}', fontsize=10)
                plt.axis('off')
        
        current_row += gsam_rows
    
    # Plot FLUX images
    if flux_images:
        # FLUX title
        plt.subplot(total_rows, 1, current_row)
        plt.text(0.5, 0.5, f'FLUX Experiment {experiment_id}', 
                ha='center', va='center', fontsize=16, fontweight='bold')
        plt.axis('off')
        current_row += 1
        
        # FLUX images
        for i, img_path in enumerate(flux_images):
            img = visualizer.load_image_safely(img_path)
            if img is not None:
                plt.subplot(flux_rows, max_cols, i + 1)
                plt.imshow(img)
                plt.title(f'FLUX: {img_path.name}', fontsize=10)
                plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def analyze_image_properties(visualizer):
    """
    Analyze properties of images across all experiments
    """
    if not visualizer.experiment_data:
        print("No experiment data found. Run find_matching_experiments() first.")
        return
    
    image_stats = []
    
    for exp_id, data in visualizer.experiment_data.items():
        # Analyze GSAM images
        for img_path in data['gsam_files']['images'][:5]:  # Sample first 5 images
            img = visualizer.load_image_safely(img_path)
            if img is not None:
                image_stats.append({
                    'Experiment': exp_id,
                    'Method': 'GSAM',
                    'Width': img.shape[1],
                    'Height': img.shape[0],
                    'Channels': img.shape[2] if len(img.shape) > 2 else 1,
                    'File Size (KB)': img_path.stat().st_size / 1024,
                    'Filename': img_path.name
                })
        
        # Analyze FLUX images
        for img_path in data['flux_files']['images'][:5]:  # Sample first 5 images
            img = visualizer.load_image_safely(img_path)
            if img is not None:
                image_stats.append({
                    'Experiment': exp_id,
                    'Method': 'FLUX',
                    'Width': img.shape[1],
                    'Height': img.shape[0],
                    'Channels': img.shape[2] if len(img.shape) > 2 else 1,
                    'File Size (KB)': img_path.stat().st_size / 1024,
                    'Filename': img_path.name
                })
    
    if not image_stats:
        print("No image data found to analyze.")
        return
    
    df = pd.DataFrame(image_stats)
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Image Properties Analysis', fontsize=16, fontweight='bold')
    
    # 1. Image dimensions distribution
    ax1 = axes[0, 0]
    for method in ['GSAM', 'FLUX']:
        method_data = df[df['Method'] == method]
        ax1.scatter(method_data['Width'], method_data['Height'], 
                   label=method, alpha=0.7, s=50)
    ax1.set_xlabel('Width (pixels)')
    ax1.set_ylabel('Height (pixels)')
    ax1.set_title('Image Dimensions')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. File size comparison
    ax2 = axes[0, 1]
    df.boxplot(column='File Size (KB)', by='Method', ax=ax2)
    ax2.set_title('File Size Distribution')
    ax2.set_xlabel('Method')
    
    # 3. Channel distribution
    ax3 = axes[0, 2]
    channel_counts = df.groupby(['Method', 'Channels']).size().unstack(fill_value=0)
    channel_counts.plot(kind='bar', ax=ax3)
    ax3.set_title('Channel Distribution')
    ax3.set_xlabel('Method')
    ax3.set_ylabel('Count')
    ax3.legend(title='Channels')
    
    # 4. Average dimensions by experiment
    ax4 = axes[1, 0]
    avg_dims = df.groupby(['Experiment', 'Method']).agg({
        'Width': 'mean',
        'Height': 'mean'
    }).reset_index()
    
    experiments = sorted(df['Experiment'].unique())
    x_pos = np.arange(len(experiments))
    width = 0.35
    
    gsam_widths = [avg_dims[(avg_dims['Experiment'] == exp) & (avg_dims['Method'] == 'GSAM')]['Width'].values[0] 
                   if len(avg_dims[(avg_dims['Experiment'] == exp) & (avg_dims['Method'] == 'GSAM')]) > 0 else 0 
                   for exp in experiments]
    flux_widths = [avg_dims[(avg_dims['Experiment'] == exp) & (avg_dims['Method'] == 'FLUX')]['Width'].values[0] 
                   if len(avg_dims[(avg_dims['Experiment'] == exp) & (avg_dims['Method'] == 'FLUX')]) > 0 else 0 
                   for exp in experiments]
    
    ax4.bar(x_pos - width/2, gsam_widths, width, label='GSAM', alpha=0.8)
    ax4.bar(x_pos + width/2, flux_widths, width, label='FLUX', alpha=0.8)
    ax4.set_xlabel('Experiment')
    ax4.set_ylabel('Average Width (pixels)')
    ax4.set_title('Average Image Width by Experiment')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels(experiments)
    ax4.legend()
    
    # 5. File size by experiment
    ax5 = axes[1, 1]
    df.boxplot(column='File Size (KB)', by=['Experiment', 'Method'], ax=ax5)
    ax5.set_title('File Size by Experiment and Method')
    ax5.tick_params(axis='x', rotation=45)
    
    # 6. Summary statistics table
    ax6 = axes[1, 2]
    ax6.axis('tight')
    ax6.axis('off')
    
    summary_stats = df.groupby('Method').agg({
        'Width': ['mean', 'std'],
        'Height': ['mean', 'std'],
        'File Size (KB)': ['mean', 'std'],
        'Channels': 'mode'
    }).round(2)
    
    # Flatten column names
    summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]
    
    table = ax6.table(cellText=summary_stats.values,
                     rowLabels=summary_stats.index,
                     colLabels=summary_stats.columns,
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.5)
    ax6.set_title('Summary Statistics')
    
    plt.tight_layout()
    plt.show()
    
    return df

def compare_experiments_side_by_side(visualizer, exp_id1: str, exp_id2: str, max_images: int = 4):
    """
    Compare two experiments side by side
    """
    if exp_id1 not in visualizer.experiment_data or exp_id2 not in visualizer.experiment_data:
        print(f"One or both experiments not found: {exp_id1}, {exp_id2}")
        return
    
    data1 = visualizer.experiment_data[exp_id1]
    data2 = visualizer.experiment_data[exp_id2]
    
    fig, axes = plt.subplots(4, max_images, figsize=(4*max_images, 16))
    fig.suptitle(f'Experiment Comparison: {exp_id1} vs {exp_id2}', fontsize=16, fontweight='bold')
    
    # Row labels
    row_labels = [f'GSAM Exp {exp_id1}', f'FLUX Exp {exp_id1}', 
                  f'GSAM Exp {exp_id2}', f'FLUX Exp {exp_id2}']
    
    image_sets = [
        data1['gsam_files']['images'][:max_images],
        data1['flux_files']['images'][:max_images],
        data2['gsam_files']['images'][:max_images],
        data2['flux_files']['images'][:max_images]
    ]
    
    for row, (images, label) in enumerate(zip(image_sets, row_labels)):
        for col in range(max_images):
            ax = axes[row, col]
            
            if col < len(images):
                img = visualizer.load_image_safely(images[col])
                if img is not None:
                    ax.imshow(img)
                    ax.set_title(f'{images[col].name}', fontsize=10)
                else:
                    ax.text(0.5, 0.5, 'Load Error', ha='center', va='center')
            else:
                ax.text(0.5, 0.5, 'No Image', ha='center', va='center')
            
            ax.axis('off')
            
            if col == 0:
                ax.set_ylabel(label, rotation=90, va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()


In [ ]:
# Example usage of the ExperimentVisualizer
# Replace these paths with your actual experiment directories
GSAM_DIR = "/path/to/gsam/experiments"  # Replace with your GSAM experiment directory
FLUX_DIR = "/path/to/flux/experiments"  # Replace with your FLUX experiment directory

"""
To use this visualization system:

1. Update the directory paths above
2. Create the visualizer instance
3. Find matching experiments
4. Generate various visualizations

Example workflow:

# Initialize the visualizer
visualizer = ExperimentVisualizer(GSAM_DIR, FLUX_DIR)

# Find matching experiments
experiment_data = visualizer.find_matching_experiments()

# Generate overview plots
df_summary = visualizer.plot_experiment_overview()

# Analyze image properties
image_analysis = analyze_image_properties(visualizer)

# Plot individual experiment
plot_individual_experiment(visualizer, "1", max_images=6)

# Compare two experiments side by side
compare_experiments_side_by_side(visualizer, "1", "2", max_images=4)

# Access individual experiment data
for exp_id, data in visualizer.experiment_data.items():
    print(f"Experiment {exp_id}:")
    print(f"  GSAM images: {len(data['gsam_files']['images'])}")
    print(f"  FLUX images: {len(data['flux_files']['images'])}")
    print(f"  GSAM pickles: {len(data['gsam_files']['pickles'])}")
    print(f"  FLUX pickles: {len(data['flux_files']['pickles'])}")
    print()

# Load and examine pickle files
if visualizer.experiment_data:
    exp_id = list(visualizer.experiment_data.keys())[0]
    data = visualizer.experiment_data[exp_id]
    
    if data['gsam_files']['pickles']:
        pickle_data = visualizer.load_pickle_safely(data['gsam_files']['pickles'][0])
        print(f"Pickle data type: {type(pickle_data)}")
        if hasattr(pickle_data, 'keys'):
            print(f"Pickle keys: {list(pickle_data.keys())}")
"""

# Uncomment and run the following lines after updating the directory paths:

# visualizer = ExperimentVisualizer(GSAM_DIR, FLUX_DIR)
# experiment_data = visualizer.find_matching_experiments()
# df_summary = visualizer.plot_experiment_overview()
